In [2]:
import os, time
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    f1_score, precision_score, recall_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# Optional: imblearn for SMOTE inside pipelines
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except Exception:
    HAS_IMBLEARN = False

# ---------------------------
# Configuration
# ---------------------------
DATASET_CONFIGS = {
    "dataset_1": {
        "path": r"D:\ML\Neurodivergent\Minsk2020_ALS_dataset.csv",
        "name": "Dataset_1"
    },
    "dataset_2": {
        "path": r"D:\ML\Neurodivergent\Parkinsson disease.csv",
        "name": "Dataset_2"
    },
    "dataset_3": {
        "path": r"D:\ML\Neurodivergent\alzheimers_disease_data.csv",
        "name": "Dataset_3"
    }
}

RANDOM_STATE = 42
BASE_RESULTS_DIR = "/content/multi_dataset_results"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

# ---------------------------
# Helper: OneHotEncoder version-safe
# ---------------------------
def get_onehot_encoder(**kwargs):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, **kwargs)

# ---------------------------
# Dataset processing functions
# ---------------------------
def detect_target(df):
    kws = ["diagnosis", "label", "class", "target", "outcome", "result", "status"]
    for c in df.columns:
        if any(k in c.lower() for k in kws):
            return c
    return df.columns[-1]

def load_and_preprocess_dataset(dataset_path, dataset_name):
    """Load and preprocess a single dataset."""
    print(f"\n{'='*50}")
    print(f"Processing {dataset_name}")
    print(f"{'='*50}")

    df = pd.read_csv(dataset_path)
    print(f"Loaded {dataset_name}:", df.shape)

    drop_like = [c for c in df.columns if c.lower() in ["id", "name", "subjectid", "patientid", "caseid"]]
    df = df.drop(columns=drop_like, errors="ignore")

    target_col = detect_target(df)
    print(f"Target detected for {dataset_name}:", target_col)

    if df[target_col].dtype == "object":
        le = LabelEncoder()
        df[target_col] = le.fit_transform(df[target_col].astype(str))
    elif not np.issubdtype(df[target_col].dtype, np.number):
        df[target_col] = pd.factorize(df[target_col])[0]

    X = df.drop(columns=[target_col])
    y = df[target_col].values

    return X, y, target_col

# ---------------------------
# Model definitions
# ---------------------------
def get_models_and_spaces():
    return {
        "rf": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {"clf__n_estimators": [100, 200, 400], "clf__max_depth": [None, 6, 12, 20], "clf__min_samples_split": [2, 5, 10]}
        ),
        "xgb": (
            XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1),
            {"clf__n_estimators": [100, 200, 400], "clf__max_depth": [3, 6, 10], "clf__learning_rate": [0.01, 0.05, 0.1]}
        ),
        "svc": (
            SVC(probability=True, random_state=RANDOM_STATE),
            {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", "auto"]}
        ),
        "knn": (
            KNeighborsClassifier(),
            {"clf__n_neighbors": [3, 5, 7, 9], "clf__weights": ["uniform", "distance"]}
        ),
        "lr": (
            LogisticRegression(max_iter=2000),
            {"clf__C": [0.01, 0.1, 1, 10]}
        ),
        "nb": (
            GaussianNB(),
            {"clf__var_smoothing": [1e-9, 1e-8, 1e-7]}
        )
    }

# ---------------------------
# Training and evaluation functions
# ---------------------------
def create_preprocessing_pipeline(X_df):
    num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_df.columns if c not in num_cols]

    print(f"Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}")

    numeric_transformer = Pipeline([
        ("imputer", IterativeImputer(random_state=RANDOM_STATE, max_iter=10)),
        ("scaler", RobustScaler())
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", get_onehot_encoder())
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ], remainder="drop", sparse_threshold=0)

    return preprocessor

def tune_and_evaluate(model_name, estimator, param_dist, X_df, y_arr, preprocessor, n_iter=12, cv_splits=3):
    use_pipeline_cls = ImbPipeline if HAS_IMBLEARN else Pipeline

    pipe_steps = [("pre", preprocessor)]
    if HAS_IMBLEARN:
        from collections import Counter
        counts = Counter(y_arr)
        if len(counts) > 1:
            imbalance = max(counts.values()) / min(counts.values())
        else:
            imbalance = 1.0
        if imbalance > 1.5:
            pipe_steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))

    k_default = min(50, max(5, int(max(5, X_df.shape[1] * 0.5))))
    pipe_steps.append(("select", SelectKBest(score_func=f_classif, k=k_default)))
    pipe_steps.append(("clf", estimator))
    pipeline = use_pipeline_cls(steps=pipe_steps)

    rskf = RepeatedStratifiedKFold(n_splits=cv_splits, n_repeats=2, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=n_iter,
                                cv=rskf, scoring="balanced_accuracy", n_jobs=-1, random_state=RANDOM_STATE, verbose=0)
    start = time.time()
    search.fit(X_df, y_arr)
    elapsed = time.time() - start
    best = search.best_estimator_
    print(f"[{model_name}] best score: {search.best_score_:.4f} time: {elapsed:.1f}s")
    return best, search

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        y_proba = model.predict_proba(X_test)
    except Exception:
        y_proba = None

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    }

    if y_proba is not None:
        try:
            if len(np.unique(y_test)) == 2:
                metrics["roc_auc"] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                metrics["roc_auc"] = roc_auc_score(y_test, y_proba, multi_class="ovr")
        except Exception:
            metrics["roc_auc"] = None
    else:
        metrics["roc_auc"] = None

    print(f"\n=== {name.upper()} ===")
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification report:\n", classification_report(y_test, y_pred))
    return metrics

def process_single_dataset(dataset_key, dataset_config):
    dataset_name = dataset_config["name"]
    dataset_path = dataset_config["path"]

    results_dir = os.path.join(BASE_RESULTS_DIR, dataset_key)
    os.makedirs(results_dir, exist_ok=True)

    X, y, target_col = load_and_preprocess_dataset(dataset_path, dataset_name)

    X_train_df, X_hold_df, y_train, y_hold = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE,
        stratify=y if len(np.unique(y)) > 1 else None
    )

    preprocessor = create_preprocessing_pipeline(X_train_df)

    models_and_spaces = get_models_and_spaces()

    print(f"\nTuning traditional ML models for {dataset_name}...")
    tuned_models = {}
    for name, (est, space) in models_and_spaces.items():
        try:
            best_est, _ = tune_and_evaluate(name, est, space, X_train_df, y_train, preprocessor)
            tuned_models[name] = best_est
        except Exception as e:
            print(f"Failed tuning {name}: {e}")

    print(f"\nEvaluating all models for {dataset_name}...")
    all_results = []
    for name, model in tuned_models.items():
        try:
            metrics = evaluate_model(name, model, X_train_df, y_train, X_hold_df, y_hold)
            all_results.append(metrics)
        except Exception as e:
            print(f"Evaluation failed for {name}: {e}")

    if all_results:
        results_df = pd.DataFrame(all_results).sort_values(by="balanced_accuracy", ascending=False).reset_index(drop=True)
        print(f"\nSummary of all models for {dataset_name}:")
        print(results_df)

        results_df.to_csv(os.path.join(results_dir, f"{dataset_key}_results.csv"), index=False)

        best_row = results_df.iloc[0]
        best_model_name = best_row["model"]
        print(f"\n🔥 Best model for {dataset_name} is: {best_model_name}")
        print(f"Best balanced accuracy: {best_row['balanced_accuracy']:.4f}")

        import joblib
        try:
            best_pipeline = tuned_models[best_model_name]
            save_path = os.path.join(results_dir, f"{dataset_key}_best_{best_model_name}.joblib")
            joblib.dump(best_pipeline, save_path)
            print(f"Saved best sklearn pipeline to: {save_path}")
        except Exception as e:
            print(f"Saving best model failed: {e}")

        return {
            "dataset_name": dataset_name,
            "results_df": results_df,
            "best_model_name": best_model_name,
            "best_metrics": best_row,
            "tuned_models": tuned_models,
            "preprocessor": preprocessor
        }
    else:
        print(f"No models evaluated successfully for {dataset_name}")
        return None

# ---------------------------
# Main execution
# ---------------------------
def main():
    print("Starting Multi-Dataset ML Pipeline")
    print(f"Processing {len(DATASET_CONFIGS)} datasets")

    all_dataset_results = {}

    for dataset_key, dataset_config in DATASET_CONFIGS.items():
        try:
            result = process_single_dataset(dataset_key, dataset_config)
            if result:
                all_dataset_results[dataset_key] = result
        except Exception as e:
            print(f"Failed to process {dataset_key}: {e}")
            continue

    if all_dataset_results:
        print(f"\n{'='*60}")
        print("CROSS-DATASET ANALYSIS")
        print(f"{'='*60}")

        comparison_data = []
        for dataset_key, result in all_dataset_results.items():
            comparison_data.append({
                "Dataset": result["dataset_name"],
                "Best_Model": result["best_model_name"],
                "Balanced_Accuracy": result["best_metrics"]["balanced_accuracy"],
                "F1_Weighted": result["best_metrics"]["f1_weighted"],
                "ROC_AUC": result["best_metrics"].get("roc_auc", "N/A")
            })

        comparison_df = pd.DataFrame(comparison_data)
        print("\nCross-dataset comparison:")
        print(comparison_df)

        comparison_df.to_csv(os.path.join(BASE_RESULTS_DIR, "cross_dataset_comparison.csv"), index=False)

        model_performance = {}
        for dataset_key, result in all_dataset_results.items():
            for _, row in result["results_df"].iterrows():
                model_name = row["model"]
                if model_name not in model_performance:
                    model_performance[model_name] = []
                model_performance[model_name].append(row["balanced_accuracy"]) 

        print("\nAverage model performance across datasets:")
        for model, scores in model_performance.items():
            avg_score = np.mean(scores)
            std_score = np.std(scores) 
            print(f"{model}: {avg_score:.4f} ± {std_score:.4f}")

    print(f"\nPipeline completed. Results saved in: {BASE_RESULTS_DIR}")

if __name__ == "__main__":
    main()


Starting Multi-Dataset ML Pipeline
Processing 3 datasets

Processing Dataset_1
Loaded Dataset_1: (64, 135)
Target detected for Dataset_1: Diagnosis (ALS)
Numeric cols: 132, Categorical cols: 1

Tuning traditional ML models for Dataset_1...
[rf] best score: 0.7569 time: 29.5s


C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\xgboost\core.py:158: UserWarning: [19:31:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 6 is smaller than n_iter=12. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[xgb] best score: 0.7095 time: 13.4s
[svc] best score: 0.7118 time: 7.9s


C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 8 is smaller than n_iter=12. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[knn] best score: 0.6956 time: 8.9s


...

C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [32] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw



=== KNN ===
Confusion matrix:
 [[215  63]
 [ 31 121]]
Classification report:
               precision    recall  f1-score   support

           0       0.87      0.77      0.82       278
           1       0.66      0.80      0.72       152

    accuracy                           0.78       430
   macro avg       0.77      0.78      0.77       430
weighted avg       0.80      0.78      0.79       430



C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [32] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw



=== LR ===
Confusion matrix:
 [[220  58]
 [ 23 129]]
Classification report:
               precision    recall  f1-score   support

           0       0.91      0.79      0.84       278
           1       0.69      0.85      0.76       152

    accuracy                           0.81       430
   macro avg       0.80      0.82      0.80       430
weighted avg       0.83      0.81      0.82       430


=== NB ===
Confusion matrix:
 [[215  63]
 [ 30 122]]
Classification report:
               precision    recall  f1-score   support

           0       0.88      0.77      0.82       278
           1       0.66      0.80      0.72       152

    accuracy                           0.78       430
   macro avg       0.77      0.79      0.77       430
weighted avg       0.80      0.78      0.79       430


Summary of all models for Dataset_3:
  model  accuracy  balanced_accuracy  f1_weighted  precision_weighted  \
0    rf  0.951163           0.942848     0.950972            0.951064   
1   xg

C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [32] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
